# Merging and Joining in Pandas

Practice relational joins, index joins, concatenation, merge indicators, and cardinality validation.

In [1]:
import pandas as pd

customers = pd.DataFrame({'customer_id': [1, 2, 3], 'name': ['Asha', 'Ravi', 'Meera'], 'city': ['Indore', 'Pune', 'Delhi']})
orders = pd.DataFrame({'order_id': [101, 102, 103, 104], 'customer_id': [1, 2, 2, 4], 'amount': [500, 750, 250, 900]})
print('Customers:')
display(customers)
print('Orders:')
display(orders)

Customers:


,customer_id,name,city
0,1,Asha,Indore
1,2,Ravi,Pune
2,3,Meera,Delhi


Orders:


,order_id,customer_id,amount
0,101,1,500
1,102,2,750
2,103,2,250
3,104,4,900


## 1. Join types

In [2]:
print('Inner merge:')
display(orders.merge(customers, on='customer_id', how='inner'))
print('Left merge:')
display(orders.merge(customers, on='customer_id', how='left'))

Inner merge:


,order_id,customer_id,amount,name,city
0,101,1,500,Asha,Indore
1,102,2,750,Ravi,Pune
2,103,2,250,Ravi,Pune


Left merge:


,order_id,customer_id,amount,name,city
0,101,1,500,Asha,Indore
1,102,2,750,Ravi,Pune
2,103,2,250,Ravi,Pune
3,104,4,900,NaN,NaN


In [3]:
print('Right merge:')
display(orders.merge(customers, on='customer_id', how='right'))
print('Outer merge:')
display(orders.merge(customers, on='customer_id', how='outer'))

Right merge:


,order_id,customer_id,amount,name,city
0,101.0,1,500.0,Asha,Indore
1,102.0,2,750.0,Ravi,Pune
2,103.0,2,250.0,Ravi,Pune
3,NaN,3,NaN,Meera,Delhi


Outer merge:


,order_id,customer_id,amount,name,city
0,101.0,1,500.0,Asha,Indore
1,102.0,2,750.0,Ravi,Pune
2,103.0,2,250.0,Ravi,Pune
3,NaN,3,NaN,Meera,Delhi
4,104.0,4,900.0,NaN,NaN


## 2. Detect unmatched keys

In [4]:
reconciled = orders.merge(customers, on='customer_id', how='outer', indicator=True)
display(reconciled)
reconciled['_merge'].value_counts()

,order_id,customer_id,amount,name,city,_merge
0,101.0,1,500.0,Asha,Indore,both
1,102.0,2,750.0,Ravi,Pune,both
2,103.0,2,250.0,Ravi,Pune,both
3,NaN,3,NaN,Meera,Delhi,right_only
4,104.0,4,900.0,NaN,NaN,left_only


_merge
both          3
left_only     1
right_only    1
Name: count, dtype: int64

## 3. Validate cardinality

In [5]:
validated = orders.merge(customers, on='customer_id', how='left', validate='many_to_one')
print('Order rows:', len(orders), '| Merged rows:', len(validated))
validated

Order rows: 4 | Merged rows: 4


,order_id,customer_id,amount,name,city
0,101,1,500,Asha,Indore
1,102,2,750,Ravi,Pune
2,103,2,250,Ravi,Pune
3,104,4,900,NaN,NaN


## 4. Different key names and suffixes

In [6]:
left = pd.DataFrame({'client_id': [1, 2], 'status': ['active', 'new']})
right = pd.DataFrame({'customer_id': [1, 2], 'status': ['verified', 'pending']})
left.merge(right, left_on='client_id', right_on='customer_id', suffixes=('_client', '_customer'))

,client_id,status_client,customer_id,status_customer
0,1,active,1,verified
1,2,new,2,pending


## 5. Index-based join

In [7]:
customers_indexed = customers.set_index('customer_id')
orders.join(customers_indexed, on='customer_id', how='left')

,order_id,customer_id,amount,name,city
0,101,1,500,Asha,Indore
1,102,2,750,Ravi,Pune
2,103,2,250,Ravi,Pune
3,104,4,900,NaN,NaN


## 6. Concatenate rows or columns

In [8]:
jan = pd.DataFrame({'order_id': [1, 2], 'amount': [100, 200]})
feb = pd.DataFrame({'order_id': [3, 4], 'amount': [150, 250]})
print('Stack rows:')
display(pd.concat([jan, feb], ignore_index=True))
print('Align columns by index:')
display(pd.concat([jan.set_index('order_id'), feb.set_index('order_id')], axis=1))

Stack rows:


,order_id,amount
0,1,100
1,2,200
2,3,150
3,4,250


Align columns by index:


,amount,amount
order_id,,
1,100.0,NaN
2,200.0,NaN
3,NaN,150.0
4,NaN,250.0


## 7. Composite keys and customer summaries

In [9]:
monthly = pd.DataFrame({'customer_id': [1, 1, 2], 'month': ['Jan', 'Feb', 'Jan'], 'amount': [100, 200, 150]})
profiles = pd.DataFrame({'customer_id': [1, 2], 'tier': ['Gold', 'Silver']})
print('Composite-key example:')
print(monthly.merge(profiles, on='customer_id', how='left'))
print('Spend per customer:')
display(orders.groupby('customer_id', as_index=False).agg(total_spend=('amount', 'sum'), order_count=('order_id', 'nunique')))

Composite-key example:
   customer_id month  amount    tier
0            1   Jan     100    Gold
1            1   Feb     200    Gold
2            2   Jan     150  Silver
Spend per customer:


,customer_id,total_spend,order_count
0,1,500,1
1,2,1000,2
2,4,900,1


## Key takeaways

- `merge()` matches keys, `join()` is handy for index-based combinations, and `concat()` stacks or aligns objects.
- Duplicate keys can multiply rows; use `validate=` to catch unexpected relationships.
- Use `indicator=True` to investigate unmatched records.